In [1]:
import openai
import re
import time
import json

import numpy as np

from tqdm import tqdm
from pprint import pprint
from tenacity import retry, stop_after_attempt, wait_chain, wait_fixed

import os
from openai import AzureOpenAI

import math

import re
import math
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
import traceback

from datetime import datetime

import concurrent.futures
import os
import json
from datetime import datetime

In [2]:
def load_json(path):
    with open(path, 'r', encoding='utf-8') as reader:
        data = json.load(reader)  # Load the entire JSON file
    return data

# Load the datasets
ASDIV = load_json('../../testingDatasets/ASDIVsampled_train.json')
ADDSUB = load_json('../../testingDatasets/AddSubsampled_train.json')
AQUA = load_json('../../testingDatasets/AQuAsampled_train.json')
GSM = load_json('../../testingDatasets/GSMsampled_train.json')
MUlTIARTH = load_json('../../testingDatasets/MultiArthsampled_train.json')
SVAMP = load_json('../../testingDatasets/SVAMPsampled_train.json')

# Load Examplars
hypothesis_CoT_prompt_examples = open('../../research/GPT4_Turbo/prompt_examples/hypothesis_CoT_prompt_examples.txt').read()
hypothesis_Standard_prompt_examples = open('../../research/GPT4_Turbo/prompt_examples/hypothesis_Standard_prompt_examples.txt').read()
hypothesis_CCoT_prompt_examples = open('../../research/GPT4_Turbo/prompt_examples/hypothesis_CCoT_prompt_examples.txt').read()
CoT_prompt_examples = open('../../research/GPT4_Turbo/prompt_examples/CoT_prompt_examples.txt').read()
Standard_prompt_examples = open("../../research/GPT4_Turbo/prompt_examples/standard_prompt_examples.txt").read()
CCoT_prompt_examples = open("../../research/GPT4_Turbo/prompt_examples/CCoT_prompt_example.txt").read()

In [3]:
endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
model_name = "gpt-4o"
deployment = "gpt-4o"
subscription_key = os.environ["AZURE_OPENAI_API_KEY"]
api_version = "2024-12-01-preview"

client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=subscription_key,
)

@retry(wait=wait_chain(*[wait_fixed(3) for _ in range(3)] +
                       [wait_fixed(5) for _ in range(2)] +
                       [wait_fixed(10)]))
def completion_with_backoff_4o(messages):
    return client.chat.completions.create(
        messages=messages,
        max_tokens=1512,
        temperature=0.0,
        model=deployment
    )

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
model_name = "gpt-35-turbo"
deployment = "gpt-35-turbo"
subscription_key = os.environ["AZURE_OPENAI_API_KEY"]
api_version = "2024-12-01-preview"

client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=subscription_key,
)

# Retry logic
@retry(wait=wait_chain(*[wait_fixed(3) for _ in range(3)] +
                       [wait_fixed(5) for _ in range(2)] +
                       [wait_fixed(10)]))
def completion_with_backoff_35(messages):
    return client.chat.completions.create(
        messages=messages,
        max_tokens=1512,
        temperature=0.0,
        model=deployment
    )

In [4]:
def clean_and_truncate(value_str):
    cleaned = re.sub(r'[^\d\.\-]', '', value_str)
    try:
        num = float(cleaned)
        return num
    except ValueError:
        return None

def clean_and_truncate(value_str):
    cleaned = re.sub(r'[^\d\.\-]', '', value_str)  # Remove everything except digits, dot, minus
    try:
        return float(cleaned)
    except ValueError:
        return None

def answer_extractor(ans_model):
    # Pattern 1: Markdown-style "### Final Answer:" followed by a number in a sentence
    match = re.search(
        r'###\s*Final Answer:\s*.*?\**\$?(-?(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?)\**',
        ans_model,
        re.IGNORECASE | re.DOTALL
    )
    if match:
        value_str = match.group(1).replace(",", "")  # Remove commas
        return clean_and_truncate(value_str)
    # Pattern 2: Standard numerical answer formats
    match = re.search(
        r'(?:the answer is|final answer:)\s*\**\$?(-?(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?(?:e[+-]?\d+)?)\**',
        ans_model,
        re.IGNORECASE
    )
    if match:
        value_str = match.group(1).replace(",", "")  # Remove commas
        return clean_and_truncate(value_str)

    # Pattern 3: Multiple-choice answer (A-E)
    match = re.search(r'the answer is\s*\**([A-E])\**', ans_model, re.IGNORECASE)
    if not match:
        match = re.search(r'\b([A-E])\b', ans_model.strip()[-5:], re.IGNORECASE)

    if match:
        predicted_choice = match.group(1).upper()
        if predicted_choice in ['A', 'B', 'C', 'D', 'E']:
            return predicted_choice

    return None

def ground_truth_extractor(d, database_name):
    if database_name == "ASDIV":
        return float(re.search(r'(\d+\.?\d*)', d['answer']).group(1))
    elif database_name == "ADDSUB":
        return float(d['correct'][0])
    elif database_name == "AQUA":
        return d['correct'].strip().upper()
    elif database_name == "GSM":
        return float(d['number_answer'])
    elif database_name == "MUlTIARTH":
        return float(d['correct'][0])
    elif database_name == "SVAMP":
        return float(d['correct'])
    else:
        raise ValueError(f"Unknown dataset: {database_name}")
def question_extractor(d, database_name):
    if database_name == "ASDIV":
        return d['body'] +' '+ d['question']
    elif database_name == "ADDSUB":
        return d['question']
    elif database_name == "AQUA":
        return f"{d['question']}\nOptions:\n{"\n".join(d['options'])}"
    elif database_name == "GSM":
        return d['question']
    elif database_name == "MUlTIARTH":
        return d['question']
    elif database_name == "SVAMP":
        return d['question']
    else:
        raise ValueError(f"Unknown dataset: {database_name}")


def generate_message(question, prompt_type, examples=None, isAQUA=False):
    if examples is None:
        examples = ""

    # Define dynamic answer suffix
    if isAQUA:
        answer_suffix = "The answer is <only option letter>"
    else:
        answer_suffix = "The answer is <only value>"

    # === Prompt instruction by prompt_type ===
    if prompt_type == "HFP-CoT":
        prompt_instruction = (
            "First, write a high-level hypothesis or plan about how to solve the problem. "
            "Then think step by step through this plan to answer the question."
        )

    elif prompt_type == "HFP-Standard":
        prompt_instruction = (
            "First, write a high-level hypothesis or plan about what the solution involves. "
            "Then, answer the problem directly using that plan."
        )

    elif prompt_type == "HFP-Complex-CoT":
        prompt_instruction = (
            "First, write a high-level hypothesis or plan. "
            "Then break the problem into labeled sub-steps and solve each step carefully using that plan."
        )

    elif prompt_type == "CoT":
        prompt_instruction = (
            "Think step by step through this question to answer the question."
        )

    elif prompt_type == "Standard":
        prompt_instruction = (
            "Answer the problem directly and provide the final answer."
        )

    elif prompt_type == "Complex-CoT":
        prompt_instruction = (
            "Break the problem into labeled sub-steps and solve each carefully. "
            "Explain your thought process clearly before arriving at the final answer."
        )

    else:
        raise ValueError(f"Unknown prompt_type: {prompt_type}")

    # === Build full user prompt ===
    prompt_q = (
        examples +
        "\n\nQ: " + question +
        f"\nA: {prompt_instruction} Write your final answer as: {answer_suffix}"
    )

    # === Compose message in OpenAI chat format ===
    messages = [
        {
            "role": "system",
            "content": f"You are solving math problems using the {prompt_type} reasoning style. "
                       f"Follow the instructions and provide a valid solution. "
                       f"Write your final answer as: {answer_suffix}"
        },
        {
            "role": "user",
            "content": prompt_q
        }
    ]

    return messages



In [5]:
import os
import json
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

# === Prompt Types ===
prompt_types = [
    "HFP-CoT", "HFP-Standard", "HFP-Complex-CoT",
    "CoT", "Standard", "Complex-CoT"
]

# === Dataset Mapping ===
dataset_map = {
    "AQUA": AQUA,
    "ASDIV": ASDIV,
    "ADDSUB": ADDSUB,
    "GSM": GSM,
    "MUlTIARTH": MUlTIARTH,
    "SVAMP": SVAMP
}

# === Load Few-Shot Examples ===
prompt_examples_map = {
    "HFP-CoT": hypothesis_CoT_prompt_examples,
    "HFP-Standard": hypothesis_Standard_prompt_examples,
    "HFP-Complex-CoT": hypothesis_CCoT_prompt_examples,
    "CoT": CoT_prompt_examples,
    "Standard": Standard_prompt_examples,
    "Complex-CoT": CCoT_prompt_examples
}

# === Define Which Models to Test ===
models_to_run = {
    "gpt-3.5-turbo": completion_with_backoff_35,
    "gpt-4o": completion_with_backoff_4o,
}

# === Results Containers ===
all_results = {}
summary_results = {}

# === Process One Question ===
def process_item(item, dataset_name, prompt_type, few_shot_examples, completion_fn):
    try:
        question = question_extractor(item, dataset_name)
        ground_truth = ground_truth_extractor(item, dataset_name)
    except Exception as e:
        return {
            "question": "[ERROR extracting question]",
            "response": str(e),
            "parsed_answer": None,
            "is_correct": False,
            "ground_truth": None
        }

    isAQUA = dataset_name == "AQUA"
    messages = generate_message(
        question=question,
        prompt_type=prompt_type,
        examples=few_shot_examples,
        isAQUA=isAQUA
    )

    try:
        model_response = completion_fn(messages)
        model_output = model_response.choices[0].message.content.strip()
    except Exception as e:
        model_output = f"[ERROR]: {str(e)}"

    parsed_answer = answer_extractor(model_output)

    is_correct = False
    try:
        if isinstance(parsed_answer, str) and isinstance(ground_truth, str):
            is_correct = parsed_answer.strip().upper() == ground_truth.strip().upper()
        elif isinstance(parsed_answer, (int, float)) and isinstance(ground_truth, (int, float, str)):
            is_correct = abs(float(parsed_answer) - float(ground_truth)) < 1e-3
    except:
        is_correct = False

    return {
        "question": question,
        "ground_truth": ground_truth,
        "response": model_output,
        "parsed_answer": parsed_answer,
        "is_correct": is_correct
    }

# === Main Loop: Model → Prompt → Dataset ===
for model_name, completion_fn in models_to_run.items():
    print(f"\n====== Running for MODEL: {model_name} ======")
    all_results[model_name] = {}
    summary_results[model_name] = {}

    for prompt_type in prompt_types:
        all_results[model_name][prompt_type] = {}
        summary_results[model_name][prompt_type] = {}

        few_shot_examples = prompt_examples_map[prompt_type]

        for dataset_name, dataset in dataset_map.items():
            print(f"→ Dataset: {dataset_name} | Prompt: {prompt_type}")
            results = []
            correct = 0
            total = 0

            def process_wrapper(item):
                return process_item(item, dataset_name, prompt_type, few_shot_examples, completion_fn)

            with ThreadPoolExecutor() as executor:
                futures = [executor.submit(process_wrapper, item) for item in dataset]
                for future in tqdm(as_completed(futures), total=len(dataset), desc=f"{model_name} - {prompt_type} - {dataset_name}"):
                    result = future.result()
                    results.append(result)
                    total += 1
                    if result["is_correct"]:
                        correct += 1

            accuracy = round((correct / total) * 100, 2) if total > 0 else 0.0
            print(f"  → Accuracy: {accuracy:.2f}% ({correct}/{total})")

            summary_results[model_name][prompt_type][dataset_name] = {
                "accuracy": accuracy,
                "correct": correct,
                "total": total
            }

            all_results[model_name][prompt_type][dataset_name] = results

            # Save to file
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            output_path = f"../../research/hypTesting/logs/{model_name}_{prompt_type}_{dataset_name}_{timestamp}.json"
            os.makedirs(os.path.dirname(output_path), exist_ok=True)
            with open(output_path, 'w', encoding='utf-8') as f:
                json.dump(results, f, indent=4)

# === Final Summary ===
print("\n=== SUMMARY ACCURACY REPORT ===")
for model_name, prompts in summary_results.items():
    print(f"\nModel: {model_name}")
    for prompt_type, datasets in prompts.items():
        print(f"  Prompt: {prompt_type}")
        for dataset_name, stats in datasets.items():
            print(f"    {dataset_name:10s}: {stats['accuracy']}% ({stats['correct']}/{stats['total']})")



====== Running for MODEL: gpt-3.5-turbo ======
→ Dataset: AQUA | Prompt: HFP-CoT


gpt-3.5-turbo - HFP-CoT - AQUA: 100%|██████████| 205/205 [05:09<00:00,  1.51s/it]


  → Accuracy: 60.00% (123/205)
→ Dataset: ASDIV | Prompt: HFP-CoT


gpt-3.5-turbo - HFP-CoT - ASDIV: 100%|██████████| 205/205 [05:04<00:00,  1.48s/it]


  → Accuracy: 90.73% (186/205)
→ Dataset: ADDSUB | Prompt: HFP-CoT


...

  → Accuracy: 77.00% (154/200)
→ Dataset: MUlTIARTH | Prompt: Complex-CoT


gpt-4o - Complex-CoT - MUlTIARTH: 100%|██████████| 205/205 [06:04<00:00,  1.78s/it]


  → Accuracy: 95.61% (196/205)
→ Dataset: SVAMP | Prompt: Complex-CoT


gpt-4o - Complex-CoT - SVAMP: 100%|██████████| 205/205 [06:53<00:00,  2.02s/it]

  → Accuracy: 77.56% (159/205)

=== SUMMARY ACCURACY REPORT ===

Model: gpt-3.5-turbo
  Prompt: HFP-CoT
    AQUA      : 60.0% (123/205)
    ASDIV     : 90.73% (186/205)
    ADDSUB    : 91.5% (183/200)
    GSM       : 76.5% (153/200)
    MUlTIARTH : 95.61% (196/205)
    SVAMP     : 82.93% (170/205)
  Prompt: HFP-Standard
    AQUA      : 63.41% (130/205)
    ASDIV     : 89.27% (183/205)
    ADDSUB    : 93.0% (186/200)
    GSM       : 70.5% (141/200)
    MUlTIARTH : 95.12% (195/205)
    SVAMP     : 85.85% (176/205)
  Prompt: HFP-Complex-CoT
    AQUA      : 61.95% (127/205)
    ASDIV     : 90.24% (185/205)
    ADDSUB    : 90.5% (181/200)
    GSM       : 77.5% (155/200)
    MUlTIARTH : 96.1% (197/205)
    SVAMP     : 84.88% (174/205)
  Prompt: CoT
    AQUA      : 63.9% (131/205)
    ASDIV     : 90.73% (186/205)
    ADDSUB    : 90.5% (181/200)
    GSM       : 81.5% (163/200)
    MUlTIARTH : 97.56% (200/205)
    SVAMP     : 80.49% (165/205)
  Prompt: Standard
    AQUA      : 55.12% (113/205)
